# Landing to Bronze

Ingestão batch (full refresh) de **todas** as fontes da Landing para a Bronze, uma seção por fonte.
A landing é somente leitura (DEC-001). Contrato Bronze: DEC-008 em `docs/decisoes.md`.

**Como executar**
- `Run All` processa as fontes em sequência. Com `gravar = nao` (padrão) cada fonte é lida e validada,
  e tabelas que já existem são conferidas contra a origem, **sem gravar nada**.
- Com `gravar = sim`, cada fonte é validada antes; só então a tabela é sobrescrita (Delta, `overwriteSchema`)
  e conferida em seguida (schema, contagem, `exceptAll` nos dois sentidos e idempotência entre as duas últimas gravações).
- `exigir_idempotencia = nao` (usado pelo Workflow) mantém a conferência contra a origem e pula a comparação entre as duas
  últimas gravações, que depende de time travel (retenção de 7 dias) e não se aplica a cargas mensais com dados novos.
- Para rodar uma fonte isolada: execute as células da seção 0 e depois as células da seção da fonte.
- Uma fonte que falha fica marcada como `FALHOU` (com o erro completo) e **não é gravada**; as demais seguem.
  A seção 9 falha explicitamente se alguma fonte estiver com falha, pendente ou sem idempotência comprovada (quando exigida).
- Os outputs contêm dados confidenciais: **nunca** commite o notebook com outputs.

## 0. Configuração
Execute as quatro células desta seção antes de qualquer fonte.
A primeira remove widgets de versões anteriores do notebook (`fonte`, `csv_*`), que não são mais usados.

In [ ]:
# Widgets de versões anteriores (seleção de fonte e opções de CSV). Removidos em uma célula própria,
# porque o Databricks não permite criar widgets na mesma célula em que outro foi removido.
for _widget_antigo in ["fonte", "csv_sep", "csv_encoding", "csv_multiline"]:
    try:
        dbutils.widgets.remove(_widget_antigo)
    except Exception:
        pass

In [ ]:
import importlib.util
import re
import traceback
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StringType, StructField, StructType, TimestampType

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"

dbutils.widgets.text("landing_path", "/Volumes/v_energy/default/landing", "Caminho da landing")
dbutils.widgets.dropdown("gravar", "nao", ["nao", "sim"], "Gravar na Bronze?")
# "nao" no Workflow: a comparação entre as duas últimas gravações usa time travel, que expira em 7 dias
# (deletedFileRetentionDuration) e não se aplica a cargas mensais com dados novos. A conferência contra a origem continua.
dbutils.widgets.dropdown("exigir_idempotencia", "sim", ["sim", "nao"], "Exigir idempotência?")
landing_path = dbutils.widgets.get("landing_path").rstrip("/")

# Opções de CSV validadas nas primeiras fontes. As fixas não podem ser sobrescritas por fonte;
# as padrão só por um override explícito na célula da fonte (versionado, com motivo no commit).
OPCOES_CSV_FIXAS = {
    "header": "true",
    "inferSchema": "false",      # tudo string: a Bronze não decide tipos
    "mode": "FAILFAST",          # linha malformada interrompe a leitura, nunca é descartada em silêncio
}
OPCOES_CSV_PADRAO = {"sep": ",", "encoding": "UTF-8", "multiLine": "false"}

COLUNAS_AUDITORIA = ["source_file", "ingestion_timestamp"]
CARACTERES_INVALIDOS_DELTA = " ,;{}()\n\t="
OPERACOES_DE_ESCRITA = {"WRITE", "CREATE TABLE AS SELECT", "REPLACE TABLE AS SELECT", "CREATE OR REPLACE TABLE AS SELECT"}
MOSTRAR_AMOSTRAS = False         # True exibe 5 linhas por fonte (output confidencial: nunca commitar)

FONTES_ESPERADAS = [
    "unidades_consumidoras", "medidores", "geracao_distribuida", "ocorrencias_rede",
    "faturas", "leituras", "atendimentos", "inspecoes",
]
RESULTADOS = {}


def gravacao_habilitada():
    """Lida a cada uso: mudar o widget e rodar só uma fonte usa o valor atual."""
    return dbutils.widgets.get("gravar") == "sim"


def idempotencia_exigida():
    return dbutils.widgets.get("exigir_idempotencia") == "sim"


print("landing:", landing_path)
print("bronze :", bronze_schema)
print("gravar :", dbutils.widgets.get("gravar"))
print("exigir_idempotencia :", dbutils.widgets.get("exigir_idempotencia"))

In [ ]:
# ---------- Funções comuns a todos os formatos ----------

def col(nome):
    """Referência literal a uma coluna: um ponto no nome não vira acesso a campo aninhado."""
    return F.col("`" + nome.replace("`", "``") + "`")


def cols(nomes):
    return [col(n) for n in nomes]


def listar(caminho):
    try:
        return {f.name: f for f in dbutils.fs.ls(caminho)}
    except Exception as e:
        raise FileNotFoundError(f"Não foi possível listar '{caminho}': {e}") from e


def exigir_arquivo(caminho):
    pasta, nome = caminho.rsplit("/", 1)
    itens = listar(pasta)
    if nome not in itens:
        raise FileNotFoundError(f"'{nome}' não encontrado em {pasta}. Conteúdo: {sorted(itens)}")
    if itens[nome].size == 0:
        raise ValueError(f"'{caminho}' está vazio.")
    return itens[nome].size


def erros_de_nomes(nomes, contexto, reservadas=True):
    """Nomes de coluna que a Bronze não aceita. Nada é renomeado: o problema vira erro."""
    erros = []
    normalizados = ["" if n is None else str(n).strip().lower() for n in nomes]
    duplicados = sorted({n for n in normalizados if normalizados.count(n) > 1})
    if duplicados:
        erros.append(f"{contexto}: nomes de coluna duplicados (ignorando caixa e espaços): {duplicados}")
    sem_nome = [n for n in nomes if n is None or str(n).strip() == "" or (str(n).startswith("_c") and str(n)[2:].isdigit())]
    if sem_nome:
        erros.append(f"{contexto}: coluna sem nome no cabeçalho: {sem_nome}")
    if any(n is not None and "﻿" in str(n) for n in nomes):
        erros.append(f"{contexto}: BOM (U+FEFF) no cabeçalho: ajuste o encoding da fonte.")
    invalidos = [n for n in nomes if n is not None and any(ch in str(n) for ch in CARACTERES_INVALIDOS_DELTA)]
    if invalidos:
        erros.append(f"{contexto}: nomes incompatíveis com Delta (não são renomeados na Bronze): {invalidos}")
    if reservadas and set(nomes) & set(COLUNAS_AUDITORIA):
        erros.append(f"{contexto}: a origem já tem colunas reservadas: {set(nomes) & set(COLUNAS_AUDITORIA)}")
    return erros


def resumir(df, colunas_origem):
    """Força o parse de todas as colunas de todas as linhas (um count() simples pode pular o parse)."""
    return df.agg(
        F.count(F.lit(1)).alias("linhas"),
        F.sum(F.hash(*cols(colunas_origem))).alias("hash_soma"),
        F.countDistinct("source_file").alias("n_source_file"),
        F.sum(F.col("source_file").isNull().cast("int")).alias("source_file_nulo"),
    ).first()


def contar_duplicatas(df, colunas_origem):
    """Informativo: linhas totalmente repetidas. Nada é removido na Bronze."""
    return (
        df.groupBy(*cols(colunas_origem)).agg(F.count(F.lit(1)).alias("__n"))
        .filter(F.col("__n") > 1).agg(F.sum(F.col("__n") - 1)).first()[0] or 0
    )


def exigir_sem_erros(erros, contexto):
    assert not erros, f"{contexto}: validação falhou, nada foi gravado:\n - " + "\n - ".join(erros)


def versoes_de_escrita(tabela):
    historico = spark.sql(f"DESCRIBE HISTORY {tabela}").select("version", "operation").collect()
    return sorted(r.version for r in historico if r.operation in OPERACOES_DE_ESCRITA), historico


def conferir(tabela, ler, colunas_origem, linhas_origem):
    """Compara a tabela com a origem (multiconjunto, sem ingestion_timestamp) e as duas últimas gravações."""
    dados = colunas_origem + ["source_file"]
    df_bronze = spark.table(tabela)
    assert df_bronze.columns == colunas_origem + COLUNAS_AUDITORIA, f"Colunas da Bronze diferentes da origem: {df_bronze.columns}"

    def tipos(df):
        return [(f.name, f.dataType.simpleString()) for f in df.select(*cols(dados)).schema.fields]

    assert tipos(df_bronze) == tipos(ler()), "Tipos das colunas da Bronze diferentes dos da origem."
    assert isinstance(df_bronze.schema["ingestion_timestamp"].dataType, TimestampType), "ingestion_timestamp não é timestamp."

    n_bronze = df_bronze.count()
    assert n_bronze == linhas_origem, f"Linhas na Bronze ({n_bronze}) diferentes da origem ({linhas_origem})."
    so_na_origem = ler().select(*cols(dados)).exceptAll(df_bronze.select(*cols(dados))).count()
    so_na_bronze = df_bronze.select(*cols(dados)).exceptAll(ler().select(*cols(dados))).count()
    assert so_na_origem == 0 and so_na_bronze == 0, f"Conteúdo divergente: só na origem {so_na_origem}, só na Bronze {so_na_bronze}."
    nulos = df_bronze.filter(F.col("source_file").isNull() | F.col("ingestion_timestamp").isNull()).count()
    assert nulos == 0, f"{nulos} linha(s) com coluna de auditoria nula."

    # Idempotência: compara as duas últimas gravações
    if not idempotencia_exigida():
        idempotencia = "não exigida (exigir_idempotencia = nao)"
        print(f"[OK] {tabela}: schema, {n_bronze} linhas e conteúdo idênticos à origem | idempotência: {idempotencia}")
        return {"linhas_bronze": n_bronze, "idempotencia": idempotencia}
    versoes, historico = versoes_de_escrita(tabela)
    if not versoes:
        raise AssertionError(f"Nenhuma gravação reconhecida no histórico de {tabela}: {sorted({r.operation for r in historico})}")
    if len(versoes) < 2:
        idempotencia = "pendente: só há uma gravação; grave de novo para comprovar"
    else:
        anterior, atual = versoes[-2], versoes[-1]
        try:
            # Leituras, colunas e contagens no mesmo try: a versão fora da retenção pode falhar já na análise da consulta
            df_atual = spark.sql(f"SELECT * FROM {tabela} VERSION AS OF {atual}")
            df_anterior = spark.sql(f"SELECT * FROM {tabela} VERSION AS OF {anterior}")
            faltam = set(dados) - set(df_anterior.columns)
            if faltam:
                raise AssertionError(f"A versão {anterior} não tem as colunas {sorted(faltam)}: as gravações têm schemas diferentes.")
            a, b = df_atual.select(*cols(dados)), df_anterior.select(*cols(dados))
            so_atual, so_anterior = a.exceptAll(b).count(), b.exceptAll(a).count()
        except Exception as e:
            if "DELTA_UNSUPPORTED_TIME_TRAVEL" in str(e):
                raise AssertionError(
                    f"Idempotência não verificável: os arquivos da versão {anterior} já saíram da retenção do time travel. "
                    "Grave de novo (gravar = sim) para comparar duas gravações recentes, ou rode com exigir_idempotencia = nao."
                ) from e
            raise
        assert so_atual == 0 and so_anterior == 0, (
            f"Idempotência falhou entre as versões {anterior} e {atual}: só na atual {so_atual}, só na anterior {so_anterior}."
        )
        idempotencia = f"ok (versões {anterior} -> {atual})"
    print(f"[OK] {tabela}: schema, {n_bronze} linhas e conteúdo idênticos à origem | idempotência: {idempotencia}")
    return {"linhas_bronze": n_bronze, "idempotencia": idempotencia}


def finalizar(tabela, ler, colunas_origem, linhas_origem, resultado):
    """Só chamada depois das validações. Grava (se habilitado) e confere; sem gravar, confere a tabela existente."""
    # Gravação condicional
    if gravacao_habilitada():
        (
            ler()
            .withColumn("ingestion_timestamp", F.current_timestamp())  # na mesma cadeia do write (lazy evaluation)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela)
        )
        print(f"[OK] {tabela} gravada (full refresh).")
        # Conferência e idempotência
        try:
            resultado.update(conferir(tabela, ler, colunas_origem, linhas_origem))
        except Exception as e:
            raise AssertionError(
                f"Conferência pós-gravação falhou: {e}. A tabela JÁ foi sobrescrita: veja DESCRIBE HISTORY {tabela} "
                f"e, se preciso, RESTORE TABLE {tabela} TO VERSION AS OF <versão anterior>."
            ) from e
        resultado["status"] = "gravada e conferida"
    # Sem gravar: conferência e idempotência da tabela existente
    elif spark.catalog.tableExists(tabela):
        resultado.update(conferir(tabela, ler, colunas_origem, linhas_origem))
        resultado["status"] = "validada; tabela existente conferida (sem gravar)"
    else:
        resultado["status"] = "validada (sem gravar; tabela ainda não existe)"
    return resultado


def executar(fonte, funcao):
    """Executa uma fonte e registra o resultado. Falha é impressa por completo e marcada, nunca gravada."""
    print(f"\n========== {fonte} ==========")
    try:
        RESULTADOS[fonte] = funcao()
    except Exception as e:
        RESULTADOS[fonte] = {"status": "FALHOU", "erro": f"{type(e).__name__}: {e}"}
        traceback.print_exc()
        print(f"\n[FALHOU] {fonte}: {type(e).__name__}: {e}")
        return
    r = RESULTADOS[fonte]
    print(f"[{fonte}] {r['status']} | idempotência: {r.get('idempotencia', '-')}")


def pendente(fonte, motivo):
    RESULTADOS[fonte] = {"status": f"PENDENTE: {motivo}"}
    print(f"[PENDENTE] {fonte}: {motivo}")

In [ ]:
# ---------- CSV (um ou mais arquivos com o mesmo conjunto de colunas) ----------

def ler_csv(caminhos, opcoes):
    """Leitura da origem: colunas originais como string + source_file. Vários arquivos são unidos por nome."""
    dfs = [
        spark.read.options(**opcoes).csv(c).select("*", F.col("_metadata.file_path").alias("source_file"))
        for c in caminhos
    ]
    df = dfs[0]
    for outro in dfs[1:]:
        df = df.unionByName(outro)  # sem allowMissingColumns: coluna ausente é erro, nunca descarte
    return df


def ingerir_csv(fonte, arquivos, opcoes_extra=None):
    """arquivos: caminhos relativos à landing. opcoes_extra: override explícito de OPCOES_CSV_PADRAO."""
    opcoes_extra = opcoes_extra or {}
    proibidas = set(opcoes_extra) - set(OPCOES_CSV_PADRAO)
    if proibidas:
        raise ValueError(f"Só é possível sobrescrever {list(OPCOES_CSV_PADRAO)}; recebido {sorted(proibidas)}.")
    opcoes = {**OPCOES_CSV_FIXAS, **OPCOES_CSV_PADRAO, **opcoes_extra}
    tabela = f"{bronze_schema}.{fonte}"
    caminhos = [f"{landing_path}/{a}" for a in arquivos]
    print(f"tabela: {tabela}")
    print("opções efetivas:", {k: f"{v} ({'override' if k in opcoes_extra else 'fixa' if k in OPCOES_CSV_FIXAS else 'padrão'})" for k, v in opcoes.items()})

    # Validação prévia: existência, cabeçalho e compatibilidade entre arquivos
    erros, cabecalhos = [], {}
    for a, c in zip(arquivos, caminhos):
        tamanho = exigir_arquivo(c)
        tokens = spark.read.options(**{**opcoes, "header": "false"}).csv(c).limit(1).first()
        tokens = list(tokens) if tokens is not None else []
        lidas = spark.read.options(**opcoes).csv(c).columns
        print(f"- {a} ({tamanho} bytes): {len(lidas)} colunas {lidas}")
        if len(lidas) < 2:
            erros.append(f"{a}: apenas {len(lidas)} coluna(s): separador provavelmente errado.")
        if lidas != tokens:
            erros.append(f"{a}: cabeçalho lido (header=true) difere do cabeçalho tokenizado.")
        erros += erros_de_nomes(lidas, a)
        cabecalhos[a] = lidas
    referencia = cabecalhos[arquivos[0]]
    for a in arquivos[1:]:
        if set(cabecalhos[a]) != set(referencia):
            erros.append(
                f"{a}: colunas incompatíveis com {arquivos[0]}: faltam {sorted(set(referencia) - set(cabecalhos[a]))}, "
                f"sobram {sorted(set(cabecalhos[a]) - set(referencia))}."
            )
        elif cabecalhos[a] != referencia:
            print(f"[INFO] {a}: mesmas colunas de {arquivos[0]} em outra ordem; a consolidação é por nome.")
    exigir_sem_erros(erros, fonte)

    # Leitura
    colunas_origem = referencia
    df = ler_csv(caminhos, opcoes)
    if df.columns != colunas_origem + ["source_file"]:
        erros.append(f"Colunas após a leitura diferentes do esperado: {df.columns}")
    nao_string = [f.name for f in df.schema.fields if not isinstance(f.dataType, StringType)]
    if nao_string:
        erros.append(f"Colunas que não são string: {nao_string}")
    exigir_sem_erros(erros, fonte)

    # Auditoria: contagem, source_file e linhas físicas
    r = resumir(df, colunas_origem)
    linhas = r["linhas"]
    if linhas == 0:
        erros.append("Nenhuma linha de dados lida.")
    if r["source_file_nulo"] != 0 or r["n_source_file"] != len(caminhos):
        erros.append(f"source_file inconsistente: {r['n_source_file']} valores distintos para {len(caminhos)} arquivo(s), {r['source_file_nulo']} nulos.")
    por_arquivo = {row["source_file"]: row["count"] for row in df.groupBy("source_file").count().collect()}
    for a, c in zip(arquivos, caminhos):
        lidos = [sf for sf in por_arquivo if sf.endswith("/" + a)]
        if len(lidos) != 1:
            erros.append(f"source_file não identifica {a}: {sorted(por_arquivo)}")
            continue
        logicas = por_arquivo[lidos[0]]
        fisicas = spark.read.text(c).count() - 1  # menos o cabeçalho
        print(f"  {a}: linhas lógicas {logicas} | físicas - cabeçalho {fisicas}")
        if logicas > fisicas:
            erros.append(f"{a}: mais linhas lógicas do que físicas; leitura inconsistente.")
        elif logicas < fisicas:
            print(f"  [ATENÇÃO] {a}: {fisicas - logicas} linha(s) física(s) a mais que as lógicas "
                  "(quebra de linha entre aspas, linhas em branco ou multiLine). Confira se é esperado.")
    exigir_sem_erros(erros, fonte)

    duplicatas = contar_duplicatas(df, colunas_origem)
    print(f"linhas: {linhas} | colunas de origem: {len(colunas_origem)} | duplicatas totais (preservadas): {duplicatas}")
    if MOSTRAR_AMOSTRAS:
        display(df.limit(5))

    # Gravação condicional e conferência/idempotência: ver finalizar()
    resultado = {"linhas": linhas, "colunas": len(colunas_origem), "duplicatas": duplicatas, "arquivos": len(caminhos)}
    return finalizar(tabela, lambda: ler_csv(caminhos, opcoes), colunas_origem, linhas, resultado)

## 1. unidades_consumidoras (CSV)
`unidades_consumidoras.csv` → `v_energy.bronze.unidades_consumidoras`. Validada no Databricks (leitura, origem × Bronze, duplicatas e idempotência).

In [ ]:
# Leitura, validação, auditoria, gravação condicional e conferência/idempotência: ver ingerir_csv() e finalizar() (seção 0)
executar("unidades_consumidoras", lambda: ingerir_csv("unidades_consumidoras", ["unidades_consumidoras.csv"]))

## 2. medidores (CSV)
`medidores.csv` → `v_energy.bronze.medidores`. Validada no Databricks (leitura, origem × Bronze, duplicatas e idempotência).

In [ ]:
# Leitura, validação, auditoria, gravação condicional e conferência/idempotência: ver ingerir_csv() e finalizar() (seção 0)
executar("medidores", lambda: ingerir_csv("medidores", ["medidores.csv"]))

## 3. geracao_distribuida (CSV)
`geracao_distribuida.csv` → `v_energy.bronze.geracao_distribuida`. Validada no Databricks (leitura, origem × Bronze, duplicatas e idempotência).

In [ ]:
# Leitura, validação, auditoria, gravação condicional e conferência/idempotência: ver ingerir_csv() e finalizar() (seção 0)
executar("geracao_distribuida", lambda: ingerir_csv("geracao_distribuida", ["geracao_distribuida.csv"]))

## 4. ocorrencias_rede (CSV)
`ocorrencias_rede.csv` → `v_energy.bronze.ocorrencias_rede`. Opções de leitura padrão, sem override.

In [ ]:
# Leitura, validação, auditoria, gravação condicional e conferência/idempotência: ver ingerir_csv() e finalizar() (seção 0)
executar("ocorrencias_rede", lambda: ingerir_csv("ocorrencias_rede", ["ocorrencias_rede.csv"]))

## 5. faturas (CSV)
`faturas.csv` → `v_energy.bronze.faturas`. Override `sep=";"`, exclusivo desta fonte. As demais opções seguem o padrão.

In [ ]:
# Leitura, validação, auditoria, gravação condicional e conferência/idempotência: ver ingerir_csv() e finalizar() (seção 0)
executar("faturas", lambda: ingerir_csv("faturas", ["faturas.csv"], {"sep": ";"}))

## 6. leituras (CSV, pasta com arquivos anuais)
Todos os CSVs de `leituras/` → `v_energy.bronze.leituras`. Antes de consolidar, o notebook exige o número esperado de
arquivos, recusa itens que não sejam CSV (nada é ignorado em silêncio) e exige o **mesmo conjunto de colunas** em todos
os arquivos. A união é por nome, sem descartar colunas.

In [ ]:
LEITURAS_PASTA = "leituras"
LEITURAS_ARQUIVOS_ESPERADOS = 5


def arquivos_leituras():
    itens = listar(f"{landing_path}/{LEITURAS_PASTA}")
    csvs = sorted(n for n in itens if not n.endswith("/") and n.lower().endswith(".csv"))
    outros = sorted(n for n in itens if n not in csvs)
    print(f"{LEITURAS_PASTA}/: {len(csvs)} CSV(s) {csvs}")
    if outros:
        raise ValueError(f"Itens que não são CSV em {LEITURAS_PASTA}/ (não são ignorados em silêncio): {outros}")
    if len(csvs) != LEITURAS_ARQUIVOS_ESPERADOS:
        raise ValueError(f"Esperados {LEITURAS_ARQUIVOS_ESPERADOS} CSVs em {LEITURAS_PASTA}/, encontrados {len(csvs)}.")
    return [f"{LEITURAS_PASTA}/{n}" for n in csvs]


# Leitura, validação, auditoria, gravação condicional e conferência/idempotência: ver ingerir_csv() e finalizar() (seção 0)
executar("leituras", lambda: ingerir_csv("leituras", arquivos_leituras()))

## 7. atendimentos (JSON)
`atendimentos.json` → `v_energy.bronze.atendimentos`, preservando a estrutura aninhada original. Os valores primitivos
são lidos como string (`primitivesAsString`), e structs e arrays são mantidos como estão.

`multiLine = True`: o diagnóstico real mostrou um array JSON em documento único.

**Verificação sem perdas** (antes de gravar): o arquivo é relido com o módulo `json` do Python, guardando números como o
texto literal do arquivo, e comparado registro a registro com o que o Spark leu. Também lista caminhos cujo tipo varia entre
objeto/array e valor simples, que o Spark converteria em texto. Se houver qualquer divergência, a fonte falha e nada é gravado.

In [ ]:
ATENDIMENTOS = {"arquivo": "atendimentos.json", "multiLine": True}  # array JSON em documento único (diagnóstico real)
LIMITE_VERIFICACAO_JSON_BYTES = 300 * 1024 * 1024  # a verificação carrega o arquivo no driver
caminho_atendimentos = f"{landing_path}/{ATENDIMENTOS['arquivo']}"
OPCOES_JSON_FIXAS = {"mode": "FAILFAST", "primitivesAsString": "true"}


def texto_multiline(valor):
    """Aceita True/False ou 'true'/'false'; devolve o texto usado pelo Spark."""
    return {True: "true", False: "false"}.get(valor, valor)


def ler_json_bruto(caminho, multiline):
    return spark.read.options(**OPCOES_JSON_FIXAS, multiLine=multiline).json(caminho)


def ler_json(caminho, multiline):
    return ler_json_bruto(caminho, multiline).select("*", F.col("_metadata.file_path").alias("source_file"))


def erros_de_schema_json(tipo, contexto):
    """Percorre structs e arrays: nomes aninhados válidos e folhas string (contrato da Bronze)."""
    erros = []
    if isinstance(tipo, StructType):
        erros += erros_de_nomes(tipo.fieldNames(), contexto, reservadas=False)
        for f in tipo.fields:
            erros += erros_de_schema_json(f.dataType, f"{contexto}.{f.name}")
    elif isinstance(tipo, ArrayType):
        erros += erros_de_schema_json(tipo.elementType, f"{contexto}[]")
    elif isinstance(tipo, MapType):
        erros.append(f"{contexto}: MapType não é suportado pelas conferências (exceptAll).")
    elif not isinstance(tipo, StringType):
        erros.append(f"{contexto}: tipo {tipo.simpleString()}, esperado string.")
    return erros


def diagnosticar_json(caminho):
    """Não grava nada. Mostra a estrutura física e testa as duas leituras com FAILFAST."""
    tamanho = exigir_arquivo(caminho)
    inicio = dbutils.fs.head(caminho, 4096).lstrip("﻿")
    primeiro = next((ch for ch in inicio if not ch.isspace()), "")
    texto = spark.read.text(caminho)
    fisicas = texto.count()
    nao_vazias = texto.filter(F.trim(F.col("value")) != "").count()
    print(f"{caminho}: {tamanho} bytes | 1º caractere não branco: {primeiro!r} | linhas físicas: {fisicas} | não vazias: {nao_vazias}")
    lidos = {}
    for ml in ("false", "true"):
        try:
            df = ler_json_bruto(caminho, ml)
            n = df.agg(F.count(F.lit(1)), F.sum(F.hash(*cols(df.columns)))).first()[0]
            lidos[ml] = n
            print(f"- multiLine={ml}: OK, {n} registro(s), {len(df.columns)} campo(s) no nível superior")
        except Exception as e:
            print(f"- multiLine={ml}: falhou ({type(e).__name__}: {str(e).splitlines()[0][:300]})")
    if primeiro == "{" and lidos.get("false") == nao_vazias:
        sugestao = '"false" (JSON Lines: um objeto por linha)'
    elif primeiro == "[" and "true" in lidos:
        sugestao = '"true" (array JSON em documento único)'
    elif primeiro == "{" and "true" in lidos and "false" not in lidos:
        sugestao = '"true" (objeto em várias linhas)'
    else:
        sugestao = "indefinida: estrutura fora dos casos previstos, avaliar manualmente"
    print("Sugestão para ATENDIMENTOS['multiLine']:", sugestao)
    escolhido = texto_multiline(ATENDIMENTOS["multiLine"]) or next((ml for ml in ("false", "true") if ml in lidos), None)
    if escolhido:
        print(f"Schema com multiLine={escolhido} (nomes de campos; output confidencial):")
        ler_json_bruto(caminho, escolhido).printSchema()


def ingerir_json(fonte, caminho, multiline):
    multiline = texto_multiline(multiline)
    if multiline not in ("false", "true"):
        raise ValueError(f"multiLine deve ser 'false' ou 'true', recebido {multiline!r}.")
    tabela = f"{bronze_schema}.{fonte}"
    print(f"tabela: {tabela}")
    print("opções efetivas:", {**OPCOES_JSON_FIXAS, "multiLine": multiline})
    exigir_arquivo(caminho)

    # Leitura e validação do schema
    bruto = ler_json_bruto(caminho, multiline)
    colunas_origem = bruto.columns
    erros = []
    if "_corrupt_record" in colunas_origem:
        erros.append("Coluna _corrupt_record presente: registros malformados.")
    erros += erros_de_nomes(colunas_origem, fonte)
    for f in bruto.schema.fields:
        erros += erros_de_schema_json(f.dataType, f"{fonte}.{f.name}")
    exigir_sem_erros(erros, fonte)

    # Auditoria: registros, source_file e estrutura
    df = ler_json(caminho, multiline)
    r = resumir(df, colunas_origem)
    linhas = r["linhas"]
    if linhas == 0:
        erros.append("Nenhum registro lido.")
    if r["n_source_file"] != 1 or r["source_file_nulo"] != 0:
        erros.append(f"source_file inconsistente: {r['n_source_file']} valores distintos, {r['source_file_nulo']} nulos.")
    elif not df.select("source_file").first()[0].endswith("/" + caminho.rsplit("/", 1)[1]):
        erros.append("source_file não aponta para o arquivo de origem.")
    if multiline == "false":
        nao_vazias = spark.read.text(caminho).filter(F.trim(F.col("value")) != "").count()
        print(f"registros: {linhas} | linhas não vazias: {nao_vazias}")
        if linhas != nao_vazias:
            erros.append(f"JSON Lines com {linhas} registros em {nao_vazias} linhas não vazias: estrutura fora do previsto.")
    exigir_sem_erros(erros, fonte)

    duplicatas = contar_duplicatas(df, colunas_origem)
    print(f"registros: {linhas} | campos no nível superior: {len(colunas_origem)} | duplicatas totais (preservadas): {duplicatas}")
    if MOSTRAR_AMOSTRAS:
        display(df.limit(5))
    resultado = {"linhas": linhas, "colunas": len(colunas_origem), "duplicatas": duplicatas, "arquivos": 1}
    # Gravação condicional e conferência/idempotência
    return finalizar(tabela, lambda: ler_json(caminho, multiline), colunas_origem, linhas, resultado)


class _NumeroLiteral(str):
    """Número JSON guardado como o texto exato do arquivo (sem passar por float)."""


def _canonico(v):
    """Forma comparável: chaves com null ou ausentes são equivalentes; booleanos como o Spark os lê."""
    if isinstance(v, dict):
        return {k: _canonico(x) for k, x in v.items() if x is not None}
    if isinstance(v, list):
        return [_canonico(x) for x in v]
    if isinstance(v, bool):
        return "true" if v else "false"
    return None if v is None else str(v)


def _tipos_por_caminho(v, caminho, acc):
    if isinstance(v, dict):
        acc.setdefault(caminho, set()).add("objeto")
        for k, x in v.items():
            _tipos_por_caminho(x, f"{caminho}.{k}", acc)
    elif isinstance(v, list):
        acc.setdefault(caminho, set()).add("array")
        for x in v:
            _tipos_por_caminho(x, f"{caminho}[]", acc)
    elif v is not None:
        tipo = "número" if isinstance(v, _NumeroLiteral) else "bool" if isinstance(v, bool) else "string"
        acc.setdefault(caminho, set()).add(tipo)


def verificar_json_sem_perdas(caminho, multiline):
    """Não grava nada. Compara o Spark com uma releitura independente (módulo json). Imprime só caminhos e contagens."""
    import json as _json
    from collections import Counter

    tamanho = exigir_arquivo(caminho)
    if tamanho > LIMITE_VERIFICACAO_JSON_BYTES:
        return {"ok": False, "motivo": f"arquivo com {tamanho} bytes acima do limite da verificação no driver"}
    with open(caminho, encoding="utf-8-sig") as fh:
        documento = _json.load(fh, parse_int=_NumeroLiteral, parse_float=_NumeroLiteral, parse_constant=_NumeroLiteral)
    registros = documento if isinstance(documento, list) else [documento]

    tipos = {}
    for r in registros:
        _tipos_por_caminho(r, "$", tipos)
    mistos = sorted(c for c, t in tipos.items() if {"objeto", "array"} & t and len(t) > 1)

    def chave(r):
        return _json.dumps(_canonico(r), sort_keys=True, ensure_ascii=False)

    origem = Counter(chave(r) for r in registros)
    lidos = Counter(chave(r.asDict(recursive=True)) for r in ler_json_bruto(caminho, texto_multiline(multiline)).collect())
    so_origem, so_spark = sum((origem - lidos).values()), sum((lidos - origem).values())

    print(f"registros: arquivo {len(registros)} | Spark {sum(lidos.values())}")
    print(f"registros só no arquivo: {so_origem} | só no Spark: {so_spark}")
    print(f"caminhos com tipo misto (objeto/array e valor simples): {mistos or 'nenhum'}")
    ok = so_origem == 0 and so_spark == 0 and not mistos
    print("[OK] Leitura sem perdas nem conversões." if ok else "[FALHOU] A leitura do Spark não reproduz o arquivo.")
    return {"ok": ok, "motivo": None if ok else
            f"{so_origem} registro(s) só no arquivo, {so_spark} só no Spark, caminhos mistos: {mistos}"}


# ---------- Execução da seção 7 ----------
# Diagnóstico (somente leitura)
try:
    diagnosticar_json(caminho_atendimentos)
except Exception as e:
    print(f"[FALHOU] diagnóstico de atendimentos: {type(e).__name__}: {e}")

# Verificação sem perdas (somente leitura)
VERIFICACAO_JSON = {"ok": False, "motivo": "verificação não executada"}
if ATENDIMENTOS["multiLine"] is not None:
    try:
        VERIFICACAO_JSON = verificar_json_sem_perdas(caminho_atendimentos, ATENDIMENTOS["multiLine"])
    except Exception as e:
        VERIFICACAO_JSON = {"ok": False, "motivo": f"{type(e).__name__}: {e}"}
        print(f"[FALHOU] verificação sem perdas de atendimentos: {type(e).__name__}: {e}")


# Gravação condicional: só com a verificação sem perdas aprovada
def _ingerir_atendimentos():
    if not VERIFICACAO_JSON["ok"]:
        raise AssertionError(f"Verificação sem perdas não aprovada, nada foi gravado: {VERIFICACAO_JSON['motivo']}")
    return ingerir_json("atendimentos", caminho_atendimentos, ATENDIMENTOS["multiLine"])


if ATENDIMENTOS["multiLine"] is None:
    pendente("atendimentos", "definir ATENDIMENTOS['multiLine'] a partir do diagnóstico acima")
else:
    executar("atendimentos", _ingerir_atendimentos)

## 8. inspecoes (XLSX, três abas)
`inspecoes.xlsx` → uma tabela Delta por aba, lida com o leitor nativo `spark.read.format("excel")` do Databricks
(sem `openpyxl`, que não existe no Serverless, e sem instalar nada), com `headerRows=1` e `dataAddress` = nome da aba.

A documentação do leitor não descreve inferência de tipos, células vazias, datas nem cabeçalhos. Por isso cada aba é
**conferida célula a célula** contra uma leitura independente do XML da planilha (biblioteca padrão), na mesma ordem:
- `exato`: o texto do Spark é igual ao valor guardado no arquivo;
- `equivalente`: mesma informação em outra representação (número `1` × `1.0`, data serial × data ISO, booleano);
- `divergente`: qualquer outra diferença, inclusive em texto (por exemplo, zeros à esquerda). Uma divergência reprova a aba.

**Leitor aprovado** (`modo = "schema_string"`) após o diagnóstico real sem divergências nas três abas. Mesmo assim,
a gravação de cada aba repete a conferência e falha se houver divergência.

In [ ]:
# Diagnóstico de estrutura (somente leitura, biblioteca padrão)
import zipfile
import xml.etree.ElementTree as ET

_NS_XLSX = {
    "m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}


def diagnosticar_xlsx_sem_dependencias(caminho):
    """Não grava nada e não lê valores de células: só a estrutura do arquivo."""
    tamanho = exigir_arquivo(caminho)
    print(f"{caminho}: {tamanho} bytes")
    with zipfile.ZipFile(caminho) as z:
        partes = set(z.namelist())
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        alvos = {r.get("Id"): r.get("Target") for r in rels}
        abas = workbook.find("m:sheets", _NS_XLSX)
        print(f"abas ({len(abas)}):")
        for aba in abas:
            alvo = alvos.get(aba.get(f"{{{_NS_XLSX['r']}}}id"), "")
            parte = alvo.lstrip("/") if alvo.startswith("/") else f"xl/{alvo}"
            dimensao = None
            if parte in partes:
                with z.open(parte) as fh:
                    for _, el in ET.iterparse(fh, events=("start",)):
                        if el.tag.endswith("}dimension"):
                            dimensao = el.get("ref")
                            break
                        if el.tag.endswith("}sheetData"):
                            break
            print(f"- {aba.get('name')!r} | estado: {aba.get('state', 'visible')} | parte: {parte} "
                  f"({'existe' if parte in partes else 'AUSENTE'}) | dimensão declarada: {dimensao}")
        print("sharedStrings.xml:", "xl/sharedStrings.xml" in partes, "| styles.xml:", "xl/styles.xml" in partes)
    print("openpyxl no ambiente:", importlib.util.find_spec("openpyxl") is not None)
    try:
        spark.read.format("excel").load(caminho).schema
        print("Spark: formato 'excel' aceito neste ambiente (não usado; avaliar semântica antes).")
    except Exception as e:
        print(f"Spark: formato 'excel' indisponível ({type(e).__name__}: {str(e).splitlines()[0][:200]})")


try:
    diagnosticar_xlsx_sem_dependencias(f"{landing_path}/inspecoes.xlsx")
except Exception as e:
    print(f"[FALHOU] diagnóstico sem dependências de inspecoes: {type(e).__name__}: {e}")


# ---------- Configuração e funções da seção 8 ----------
import posixpath
from collections import Counter
from datetime import datetime as _datetime, timedelta
from decimal import Decimal, InvalidOperation

INSPECOES = {
    "arquivo": "inspecoes.xlsx",
    "abas": {"ago-set": "inspecoes_ago_set", "out-nov": "inspecoes_out_nov", "dez": "inspecoes_dez"},
    "modo": "schema_string",     # "schema_string": schema explícito todo string | "inferido_cast": tipos inferidos -> string
    "leitor_aprovado": True,     # aprovado: diagnóstico real sem divergências nas três abas, nos dois modos
}
caminho_inspecoes = f"{landing_path}/{INSPECOES['arquivo']}"

_M = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
_R = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
_FORMATOS_DATA_EMBUTIDOS = set(range(14, 23)) | set(range(27, 37)) | set(range(45, 48)) | set(range(50, 59))
_FORMATOS_DATA_SPARK = ("%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S.%f", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d")


# ---------- Oráculo: leitura independente do XML da aba (biblioteca padrão) ----------

def _indice_coluna(ref):
    n = 0
    for ch in ref:
        if not ch.isalpha():
            break
        n = n * 26 + (ord(ch.upper()) - 64)
    return n - 1


def _texto_rico(el):
    """Texto de <si>/<is>: <t> direto ou runs <r><t>, sem anotações fonéticas (<rPh>)."""
    partes = [t.text or "" for t in el.findall(f"{_M}t")]
    partes += [t.text or "" for r in el.findall(f"{_M}r") for t in r.findall(f"{_M}t")]
    return "".join(partes)


def _formato_eh_data(codigo):
    limpo = re.sub(r'"[^"]*"|\[[^\]]*\]|\\.', "", codigo or "")
    return bool(re.search(r"[ydhs]", limpo, re.I))


def _estilos_de_data(z):
    if "xl/styles.xml" not in z.namelist():
        return set()
    estilos = ET.fromstring(z.read("xl/styles.xml"))
    personalizados = {int(n.get("numFmtId")): n.get("formatCode") for n in estilos.iter(f"{_M}numFmt")}
    xfs = estilos.find(f"{_M}cellXfs")
    datas = set()
    for i, xf in enumerate(xfs.findall(f"{_M}xf") if xfs is not None else []):
        fmt = int(xf.get("numFmtId", 0))
        if fmt in _FORMATOS_DATA_EMBUTIDOS or (fmt in personalizados and _formato_eh_data(personalizados[fmt])):
            datas.add(i)
    return datas


def ler_aba_xml(caminho, aba):
    """Cabeçalho, linhas de dados e metadados. Cada célula é None ou (tipo, texto guardado no arquivo)."""
    with zipfile.ZipFile(caminho) as z:
        partes = set(z.namelist())
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        pr = workbook.find(f"{_M}workbookPr")
        base_datas = _datetime(1904, 1, 1) if pr is not None and pr.get("date1904") in ("1", "true") else _datetime(1899, 12, 30)
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        alvos = {r.get("Id"): r.get("Target") for r in rels}
        folha = next((s for s in workbook.iter(f"{_M}sheet") if s.get("name") == aba), None)
        if folha is None:
            raise KeyError(f"Aba {aba!r} não existe no arquivo.")
        alvo = alvos[folha.get(f"{_R}id")]
        parte = alvo.lstrip("/") if alvo.startswith("/") else posixpath.normpath(f"xl/{alvo}")
        compartilhadas = []
        if "xl/sharedStrings.xml" in partes:
            sst = ET.fromstring(z.read("xl/sharedStrings.xml"))
            compartilhadas = [_texto_rico(si) for si in sst.findall(f"{_M}si")]
        estilos_data = _estilos_de_data(z)

        celulas, mescladas, dimensao = {}, 0, None
        linha, coluna = -1, -1
        with z.open(parte) as fh:
            for evento, el in ET.iterparse(fh, events=("start", "end")):
                if evento == "start":
                    if el.tag == f"{_M}row":
                        linha = int(el.get("r")) - 1 if el.get("r") else linha + 1
                        coluna = -1
                    continue
                if el.tag == f"{_M}dimension":
                    dimensao = el.get("ref")
                elif el.tag == f"{_M}mergeCell":
                    mescladas += 1
                elif el.tag == f"{_M}c":
                    ref = el.get("r")
                    coluna = _indice_coluna(ref) if ref else coluna + 1
                    tipo = el.get("t", "n")
                    v = el.find(f"{_M}v")
                    texto = v.text if v is not None else None
                    if tipo == "inlineStr":
                        inline = el.find(f"{_M}is")
                        valor = ("string", _texto_rico(inline)) if inline is not None else None
                    elif texto is None:
                        valor = None
                    elif tipo == "s":
                        valor = ("string", compartilhadas[int(texto)])
                    elif tipo == "str":
                        valor = ("string", texto)
                    elif tipo == "b":
                        valor = ("bool", texto)
                    elif tipo == "e":
                        valor = ("erro", texto)
                    elif tipo == "d":
                        valor = ("data_iso", texto)
                    else:
                        valor = ("data" if int(el.get("s", 0)) in estilos_data else "numero", texto)
                    if valor is not None:
                        celulas[(linha, coluna)] = valor
                elif el.tag == f"{_M}row":
                    el.clear()

    info = {"dimensao": dimensao, "mescladas": mescladas, "base_datas": base_datas,
            "tipos": Counter(v[0] for v in celulas.values())}
    if not celulas:
        return [], [], info
    n_linhas = max(l for l, _ in celulas) + 1
    n_colunas = max(c for _, c in celulas) + 1
    grade = [[celulas.get((l, c)) for c in range(n_colunas)] for l in range(n_linhas)]
    cabecalho = [None if v is None else v[1] for v in grade[0]]
    info["vazias"] = sum(v is None for linha_ in grade[1:] for v in linha_)
    return cabecalho, grade[1:], info


# ---------- Leitura pelo Spark e comparação célula a célula ----------

def ler_excel_aba(caminho, aba, cabecalho, modo):
    """Colunas da aba como string + source_file (lido do scan, antes de qualquer projeção)."""
    leitor = spark.read.format("excel").option("headerRows", 1).option("dataAddress", aba)
    if modo == "schema_string":
        df = leitor.schema(StructType([StructField(c, StringType(), True) for c in cabecalho])).load(caminho)
        dados = [col(c) for c in df.columns]
    elif modo == "inferido_cast":
        df = leitor.load(caminho)
        dados = [col(c).cast("string").alias(c) for c in df.columns]
    else:
        raise ValueError(f"modo desconhecido: {modo!r}")
    return df.select(*dados, F.col("_metadata.file_path").alias("source_file"))


def _como_data(texto):
    for formato in _FORMATOS_DATA_SPARK:
        try:
            return _datetime.strptime(texto.strip(), formato)
        except ValueError:
            pass
    return None


def _como_decimal(texto):
    try:
        return Decimal(texto.strip())
    except (InvalidOperation, AttributeError):
        return None


def classificar_celula(oraculo, lido, base_datas):
    if oraculo is None or oraculo == ("string", ""):
        return "exato" if lido is None else "equivalente" if lido == "" else "divergente"
    tipo, texto = oraculo
    if lido is None:
        return "divergente"
    if lido == texto:
        return "exato"
    if tipo in ("numero", "data"):
        a, b = _como_decimal(texto), _como_decimal(lido)
        if a is not None and b is not None and a == b:
            return "equivalente"
    if tipo == "data":
        lida = _como_data(lido)
        if lida is not None and abs((lida - (base_datas + timedelta(days=float(texto)))).total_seconds()) < 1:
            return "equivalente"
    if tipo == "data_iso" and _como_data(lido) is not None and _como_data(lido) == _como_data(texto):
        return "equivalente"
    if tipo == "bool" and lido.strip().lower() == ("true" if texto == "1" else "false"):
        return "equivalente"
    return "divergente"


def comparar_aba(linhas_xml, linhas_spark, cabecalho, base_datas):
    """Comparação posicional: se a ordem das linhas não for a do arquivo, o resultado é divergência, nunca falso acerto."""
    por_coluna = {c: Counter() for c in cabecalho}
    exemplos = {}
    for oraculo_l, lido_l in zip(linhas_xml, linhas_spark):
        for c, oraculo, lido in zip(cabecalho, oraculo_l, lido_l):
            classe = classificar_celula(oraculo, lido, base_datas)
            tipo = "vazia" if oraculo is None else oraculo[0]
            por_coluna[c][(classe, tipo)] += 1
            if classe == "divergente" and len(exemplos.setdefault(c, [])) < 2:
                exemplos[c].append((oraculo, lido))
    total = Counter()
    for contagem in por_coluna.values():
        for (classe, _), n in contagem.items():
            total[classe] += n
    return {"linhas_xml": len(linhas_xml), "linhas_spark": len(linhas_spark), "total": total,
            "por_coluna": por_coluna, "exemplos": exemplos}


def imprimir_comparacao(comp):
    print(f"  linhas: XML {comp['linhas_xml']} | Spark {comp['linhas_spark']} | células: {dict(comp['total'])}")
    for c, contagem in comp["por_coluna"].items():
        print(f"    {c!r}: " + ", ".join(f"{classe}/{tipo}={n}" for (classe, tipo), n in sorted(contagem.items())))
    for c, ex in comp["exemplos"].items():
        print(f"    [DIVERGENTE] {c!r}: (arquivo, Spark) = {ex}")


def aprovada(comp):
    return comp["linhas_xml"] == comp["linhas_spark"] and comp["total"]["divergente"] == 0


def diagnosticar_excel(caminho):
    """Somente leitura. Não grava nada."""
    exigir_arquivo(caminho)
    try:
        abas = [r.asDict(recursive=True) for r in
                spark.read.format("excel").option("operation", "listSheets").load(caminho).collect()]
        print("listSheets:", abas)
    except Exception as e:
        print(f"listSheets falhou: {type(e).__name__}: {str(e).splitlines()[0][:200]}")
    for aba, destino in INSPECOES["abas"].items():
        print(f"\n--- aba {aba!r} -> {bronze_schema}.{destino}")
        cabecalho, linhas_xml, info = ler_aba_xml(caminho, aba)
        print(f"  XML: {len(cabecalho)} colunas | {len(linhas_xml)} linhas de dados | dimensão declarada {info['dimensao']} | "
              f"células vazias {info.get('vazias')} | mescladas {info['mescladas']} | tipos {dict(info['tipos'])}")
        print(f"  cabeçalho: {cabecalho}")
        problemas = erros_de_nomes(cabecalho, f"aba {aba!r}")
        if problemas:
            print("  [FALHOU] cabeçalho:", problemas)
            continue
        try:
            inferido = spark.read.format("excel").option("headerRows", 1).option("dataAddress", aba).load(caminho)
            print("  schema inferido pelo Spark:", [(f.name, f.dataType.simpleString()) for f in inferido.schema.fields])
        except Exception as e:
            print(f"  leitura inferida falhou: {type(e).__name__}: {str(e).splitlines()[0][:200]}")
        for modo in ("schema_string", "inferido_cast"):
            try:
                df = ler_excel_aba(caminho, aba, cabecalho, modo)
                colunas = [c for c in df.columns if c != "source_file"]
                if colunas != cabecalho:
                    print(f"  modo {modo}: [FALHOU] colunas do Spark diferentes do cabeçalho: {colunas}")
                    continue
                linhas_spark = [list(r) for r in df.select(*cols(colunas)).collect()]
                comp = comparar_aba(linhas_xml, linhas_spark, cabecalho, info["base_datas"])
                arquivos = [r[0] for r in df.select("source_file").distinct().collect()]
                print(f"  modo {modo}: {'APROVADO' if aprovada(comp) else 'REPROVADO'} | source_file: {arquivos}")
                imprimir_comparacao(comp)
            except Exception as e:
                print(f"  modo {modo}: [FALHOU] {type(e).__name__}: {str(e).splitlines()[0][:300]}")


def ingerir_aba_excel(destino, caminho, aba, modo):
    if not re.fullmatch(r"[a-z][a-z0-9_]*", destino):
        raise ValueError(f"Nome de tabela inválido: {destino!r}")
    tabela = f"{bronze_schema}.{destino}"
    print(f"aba {aba!r} -> {tabela} | format('excel'), headerRows=1, dataAddress={aba!r}, modo={modo}")
    exigir_arquivo(caminho)

    # Validação do cabeçalho
    cabecalho, linhas_xml, info = ler_aba_xml(caminho, aba)
    erros = erros_de_nomes(cabecalho, f"aba {aba!r}")
    if not cabecalho:
        erros.append("Aba vazia.")
    exigir_sem_erros(erros, destino)

    # Leitura
    df = ler_excel_aba(caminho, aba, cabecalho, modo)
    if df.columns != cabecalho + ["source_file"]:
        erros.append(f"Colunas do Spark diferentes do cabeçalho do arquivo: {df.columns}")
    nao_string = [f.name for f in df.schema.fields if not isinstance(f.dataType, StringType)]
    if nao_string:
        erros.append(f"Colunas que não são string: {nao_string}")
    exigir_sem_erros(erros, destino)

    # Validação: comparação célula a célula com o XML
    comp = comparar_aba(linhas_xml, [list(r) for r in df.select(*cols(cabecalho)).collect()], cabecalho, info["base_datas"])
    imprimir_comparacao(comp)
    if not aprovada(comp):
        erros.append("A leitura do Spark não reproduz a planilha (contagem ou células divergentes).")
    # Auditoria: contagem e source_file
    r = resumir(df, cabecalho)
    linhas = r["linhas"]
    if linhas == 0:
        erros.append("Nenhuma linha de dados.")
    if r["n_source_file"] != 1 or r["source_file_nulo"] != 0:
        erros.append("source_file inconsistente.")
    elif not df.select("source_file").first()[0].endswith("/" + INSPECOES["arquivo"]):
        erros.append("source_file não aponta para o arquivo de origem.")
    exigir_sem_erros(erros, destino)

    duplicatas = contar_duplicatas(df, cabecalho)
    print(f"linhas: {linhas} | colunas: {len(cabecalho)} | duplicatas totais (preservadas): {duplicatas}")
    if MOSTRAR_AMOSTRAS:
        display(df.limit(5))
    resultado = {"linhas": linhas, "colunas": len(cabecalho), "duplicatas": duplicatas, "arquivos": 1,
                 "equivalentes": comp["total"]["equivalente"]}
    # Gravação condicional e conferência/idempotência
    return finalizar(tabela, lambda: ler_excel_aba(caminho, aba, cabecalho, modo), cabecalho, linhas, resultado)


# Diagnóstico com leitor nativo (somente leitura)
try:
    diagnosticar_excel(caminho_inspecoes)
except Exception as e:
    print(f"[FALHOU] diagnóstico de inspecoes: {type(e).__name__}: {e}")


# ---------- Execução da seção 8 ----------
# Gravação condicional por aba
if not INSPECOES["leitor_aprovado"]:
    pendente("inspecoes", "aprovar a leitura format('excel') após o diagnóstico (INSPECOES['leitor_aprovado'])")
else:
    for aba, destino in INSPECOES["abas"].items():
        executar(f"inspecoes/{destino}",
                 lambda aba=aba, destino=destino: ingerir_aba_excel(destino, caminho_inspecoes, aba, INSPECOES["modo"]))

## 9. Resumo
Imprime o estado de cada fonte. Output **confidencial** (contagens reais): use só no relatório privado, nunca no Git.
Falha explicitamente se alguma fonte não foi executada, falhou, está pendente ou ainda não tem idempotência comprovada.

In [ ]:
print("CONFIDENCIAL: resultados derivados do dataset. Não versionar.")
print(f"gravar = {dbutils.widgets.get('gravar')} | exigir_idempotencia = {dbutils.widgets.get('exigir_idempotencia')} | "
      f"{datetime.now().isoformat(timespec='seconds')}")
incompletas = []
for fonte in FONTES_ESPERADAS:
    chaves = [k for k in RESULTADOS if k == fonte or k.startswith(fonte + "/")]
    if not chaves:
        print(f"- {fonte}: NÃO EXECUTADA")
        incompletas.append(fonte)
        continue
    for k in chaves:
        r = RESULTADOS[k]
        print(f"- {k}: {r['status']} | linhas {r.get('linhas', '-')} | colunas {r.get('colunas', '-')} | "
              f"duplicatas {r.get('duplicatas', '-')} | idempotência {r.get('idempotencia', '-')}")
        if "erro" in r:
            print(f"    erro: {r['erro']}")
        idem = str(r.get("idempotencia", ""))
        concluida = r["status"] in ("gravada e conferida", "validada; tabela existente conferida (sem gravar)") \
            and (idem.startswith("ok") or (not idempotencia_exigida() and idem.startswith("não exigida")))
        if not concluida:
            incompletas.append(k)

assert not incompletas, f"Bronze NÃO concluída. Não executadas, com falha, pendentes ou sem idempotência: {incompletas}"
print("[OK] Todas as fontes conferidas contra a origem"
      + (" e com idempotência comprovada." if idempotencia_exigida() else " (idempotência não exigida)."))